In [ ]:
# Script calculates Contact Frequency and Number of Contacts between two different molecule types in a simuulation box
# The abbreviation "OWWW" stands for C8-WWWDK and "WWNW" stands for C8-WWNWK.

import MDAnalysis as mda
from MDAnalysis.analysis import distances
import numpy as np
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

# Load trajectory
print("Loading trajectory...")
u = mda.Universe(r".\data\mix.tpr", 
                 r".\data\mix.xtc")

mol1_sel = "resname OWWW"
mol2_sel = "resname WWNW"

print(f"Trajectory loaded: {len(u.trajectory)} frames")
print(f"OWWW atoms: {len(u.select_atoms(mol1_sel))}")
print(f"WWNW atoms: {len(u.select_atoms(mol2_sel))}\n")

# Analysis with progress bar
start_time = time.time()
min_distances = []
contact_freq = []
contact_threshold = 3.5

for frame in tqdm(u.trajectory, desc="Analyzing", unit="frame"):
    mol1 = u.select_atoms(mol1_sel)
    mol2 = u.select_atoms(mol2_sel)
    
    d = distances.distance_array(mol1.positions, mol2.positions)
    min_distances.append(np.min(d))
    contact_freq.append(np.sum(d < contact_threshold))

elapsed = time.time() - start_time

# Results
print(f"\n{'='*60}")
print(f"✅ ANALYSIS COMPLETE")
print(f"{'='*60}")
print(f"Time elapsed: {elapsed:.2f} seconds")
print(f"Frames processed: {len(u.trajectory)}")
print(f"Time per frame: {elapsed/len(u.trajectory):.4f} s")
print(f"\nResults:")
print(f"  Min distance: {np.min(min_distances):.2f} Å")
print(f"  Max distance: {np.max(min_distances):.2f} Å")
print(f"  Avg contacts: {np.mean(contact_freq):.2f}")
print(f"{'='*60}\n")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(min_distances)
axes[0].set_ylabel("Min Distance (Å)")
axes[0].set_title("OWWW-WWNW Interaction Distance")
axes[0].axhline(y=3.5, color='r', linestyle='--', alpha=0.5)

axes[1].plot(contact_freq)
axes[1].set_ylabel("Number of Contacts")
axes[1].set_xlabel("Frame")
axes[1].set_title("Contact Frequency")

plt.tight_layout()
plt.savefig("interaction_analysis.png")
print("Plot saved to: interaction_analysis.png")
plt.show()

In [ ]:
# Script calculates Interaction HEATMAP MATRIX between two different molecule types in a simuulation box
# The abbreviation "OWWW" stands for C8-WWWDK and "WWNW" stands for C8-WWNWK.

import MDAnalysis as mda
import numpy as np
from tqdm import tqdm
import time
from datetime import datetime
from collections import defaultdict
import os
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# CONFIGURATION
# ============================================================================
OUTPUT_DIR = r".\data\output"
CONTACT_THRESHOLD = 3.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD TRAJECTORY
# ============================================================================
print("Loading trajectory...")
u = mda.Universe(r".\data\mix.tpr", 
                 r".\data\mix.xtc")

mol1_sel = "resname OWWW and not resname SOL"
mol2_sel = "resname WWNW and not resname SOL"

mol1_atoms = u.select_atoms(mol1_sel)
mol2_atoms = u.select_atoms(mol2_sel)

print(f"✓ OWWW atoms: {len(mol1_atoms)}")
print(f"✓ WWNW atoms: {len(mol2_atoms)}")
print(f"✓ Total frames: {len(u.trajectory)}\n")

# Pre-store atom information
owww_atom_names = set(mol1_atoms.names)
wwnw_atom_names = set(mol2_atoms.names)

print(f"OWWW atom types: {sorted(owww_atom_names)}")
print(f"WWNW atom types: {sorted(wwnw_atom_names)}\n")

# ============================================================================
# FAST ANALYSIS - WITH MOLECULE TRACKING
# ============================================================================

inter_frame_data = []
intermolecular_pairs = defaultdict(int)
atom_type_contacts = defaultdict(int)
atom_molecule_source = {}  # Track which molecule each atom type is from

start_time = time.time()

print("Analyzing INTERMOLECULAR interactions with MOLECULE SOURCE...")

for frame_idx, frame in enumerate(tqdm(u.trajectory, desc="Processing frames", unit="frame")):
    mol1 = u.select_atoms(mol1_sel)
    mol2 = u.select_atoms(mol2_sel)
    
    # VECTORIZED distance calculation
    dist_matrix = cdist(mol1.positions, mol2.positions)
    
    # Find all contacts at once
    contacts = np.where(dist_matrix < CONTACT_THRESHOLD)
    
    frame_contact_count = len(contacts[0])
    inter_frame_data.append((frame_idx, frame_contact_count))
    
    # Extract atom names (vectorized)
    atom1_names = mol1.names[contacts[0]]  # From OWWW
    atom2_names = mol2.names[contacts[1]]  # From WWNW
    
    # Count pairs WITH molecule information
    for name1, name2 in zip(atom1_names, atom2_names):
        # Create pair key with molecule information
        pair_key = f"{name1}(OWWW)-{name2}(WWNW)"
        intermolecular_pairs[pair_key] += 1
        
        # Track atom source
        atom_molecule_source[name1] = "OWWW"
        atom_molecule_source[name2] = "WWNW"
        
        atom_type_contacts[f"{name1}(OWWW)"] += 1
        atom_type_contacts[f"{name2}(WWNW)"] += 1

elapsed = time.time() - start_time

# ============================================================================
# SORT DATA
# ============================================================================

sorted_inter_pairs = sorted(intermolecular_pairs.items(), 
                            key=lambda x: x[1], reverse=True)
sorted_atom_types = sorted(atom_type_contacts.items(), 
                          key=lambda x: x[1], reverse=True)

frames = np.array([x[0] for x in inter_frame_data])
contacts = np.array([x[1] for x in inter_frame_data])

# ============================================================================
# GENERATE REPORT
# ============================================================================

report = []
report.append("=" * 120)
report.append("INTERMOLECULAR INTERACTION ANALYSIS - WITH MOLECULE SOURCE")
report.append("=" * 120)
report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report.append(f"Contact threshold: {CONTACT_THRESHOLD} Å")
report.append(f"Total frames analyzed: {len(u.trajectory)}")
report.append(f"Analysis time: {elapsed:.2f} seconds ({elapsed/len(u.trajectory):.4f}s per frame)")
report.append("")

total_contacts = np.sum(contacts)

report.append("=" * 120)
report.append("ATOM TYPE INVENTORY")
report.append("=" * 120)
report.append(f"OWWW atom types: {sorted(owww_atom_names)}")
report.append(f"WWNW atom types: {sorted(wwnw_atom_names)}")
report.append("")

report.append("=" * 120)
report.append("TIME-DEPENDENT STATISTICS")
report.append("=" * 120)
report.append(f"Total OWWW-WWNW contacts: {total_contacts}")
report.append(f"Average contacts per frame: {np.mean(contacts):.2f}")
report.append(f"Max contacts in a frame: {np.max(contacts)}")
report.append(f"Min contacts in a frame: {np.min(contacts)}")
report.append(f"Std deviation: {np.std(contacts):.2f}")
report.append(f"Coefficient of variation: {(np.std(contacts)/np.mean(contacts))*100:.2f}%")
report.append("")

# Top atom types with molecule info
report.append("=" * 120)
report.append("TOP 30 MOST INTERACTIVE ATOM TYPES (WITH MOLECULE SOURCE)")
report.append("=" * 120)
report.append(f"{'Rank':<5} {'Atom (Molecule)':<30} {'Total Contacts':<20} {'% of Total':<15}")
report.append("-" * 70)

for rank, (atom_type, count) in enumerate(sorted_atom_types[:30], 1):
    percentage = (count / sum(atom_type_contacts.values())) * 100
    report.append(f"{rank:<5} {atom_type:<30} {count:<20} {percentage:.2f}%")

report.append("")

# Top intermolecular pairs with molecule info
report.append("=" * 120)
report.append("TOP 50 INTERACTION PAIRS (OWWW ↔ WWNW)")
report.append("=" * 120)
report.append(f"{'Rank':<5} {'OWWW Atom':<20} {'WWNW Atom':<20} {'Contact Count':<15} {'% of Total':<15}")
report.append("-" * 80)

for rank, (pair, count) in enumerate(sorted_inter_pairs[:50], 1):
    # Parse pair to show molecule source
    owww_atom, wwnw_atom = pair.split('-')
    percentage = (count / total_contacts) * 100
    report.append(f"{rank:<5} {owww_atom:<20} {wwnw_atom:<20} {count:<15} {percentage:.2f}%")

report.append("")

# Detailed pair information
report.append("=" * 120)
report.append("DETAILED ANALYSIS - TOP 20 PAIRS")
report.append("=" * 120)

for rank, (pair, count) in enumerate(sorted_inter_pairs[:20], 1):
    owww_atom, wwnw_atom = pair.split('-')
    percentage = (count / total_contacts) * 100
    report.append(f"\n#{rank}: {owww_atom} ↔ {wwnw_atom}")
    report.append(f"   Total contacts: {count:,}")
    report.append(f"   Percentage of all interactions: {percentage:.3f}%")
    report.append(f"   Average contacts per frame: {count / len(u.trajectory):.2f}")

report.append("")
report.append("=" * 120)

report_text = "\n".join(report)
print("\n" + report_text)

output_file = os.path.join(OUTPUT_DIR, "interaction_analysis_WITH_MOLECULE_SOURCE.txt")
with open(output_file, "w") as f:
    f.write(report_text)

print(f"\n✅ Report saved to: {output_file}")

# ============================================================================
# PLOTS WITH MOLECULE INFORMATION
# ============================================================================

print("\nGenerating plots with molecule information...")

# PLOT 1: Time-dependent interactions
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(frames, contacts, linewidth=1.5, color='darkblue', alpha=0.7)
ax.fill_between(frames, contacts, alpha=0.3, color='lightblue')

window = min(50, len(contacts)//10)
moving_avg = np.convolve(contacts, np.ones(window)/window, mode='valid')
frames_avg = frames[window-1:]
ax.plot(frames_avg, moving_avg, linewidth=2.5, color='red', 
        label=f'Moving avg (window={window})', linestyle='--')

ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Contacts', fontsize=12, fontweight='bold')
ax.set_title('OWWW ↔ WWNW Interactions Over Time', fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10)

plt.tight_layout()
plot_file_1 = os.path.join(OUTPUT_DIR, "01_time_dependent_interactions.png")
plt.savefig(plot_file_1, dpi=300, bbox_inches='tight')
print(f"✅ Plot 1: {plot_file_1}")
plt.close()

# PLOT 2: Top 40 pairs with color coding by molecule
fig, ax = plt.subplots(figsize=(13, 12))

top_40_pairs = sorted_inter_pairs[:40]
pair_labels = []
pair_counts = []
pair_colors = []

for pair, count in top_40_pairs:
    owww_atom, wwnw_atom = pair.split('-')
    pair_labels.append(f"{owww_atom}(OWWW)\n↔\n{wwnw_atom}(WWNW)")
    pair_counts.append(count)
    
    # Color based on which OWWW atom
    hue = (ord(owww_atom[0]) - ord('A')) / 26
    pair_colors.append(plt.cm.hsv(hue))

bars = ax.barh(range(len(pair_labels)), pair_counts, color=pair_colors, edgecolor='black', linewidth=0.7)

ax.set_yticks(range(len(pair_labels)))
ax.set_yticklabels(pair_labels, fontsize=8)
ax.set_xlabel('Contact Count', fontsize=12, fontweight='bold')
ax.set_title('Top 40 Interaction Pairs\n(OWWW atoms with WWNW atoms)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plot_file_2 = os.path.join(OUTPUT_DIR, "02_top_40_pairs_with_molecules.png")
plt.savefig(plot_file_2, dpi=300, bbox_inches='tight')
print(f"✅ Plot 2: {plot_file_2}")
plt.close()

# PLOT 3: Atom types separated by molecule
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# OWWW atoms
owww_contacts = {k: v for k, v in atom_type_contacts.items() if 'OWWW' in k}
owww_sorted = sorted(owww_contacts.items(), key=lambda x: x[1], reverse=True)[:15]
owww_names = [x[0].replace('(OWWW)', '') for x in owww_sorted]
owww_counts = [x[1] for x in owww_sorted]

colors_owww = plt.cm.Blues(np.linspace(0.4, 0.9, len(owww_counts)))
bars1 = axes[0].barh(range(len(owww_names)), owww_counts, color=colors_owww, edgecolor='black', linewidth=1)

axes[0].set_yticks(range(len(owww_names)))
axes[0].set_yticklabels(owww_names, fontsize=11, fontweight='bold')
axes[0].set_xlabel('Total Contacts', fontsize=12, fontweight='bold')
axes[0].set_title('Top OWWW Atoms (Most Interactive)', fontsize=13, fontweight='bold', pad=15)
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

for i, (bar, count) in enumerate(zip(bars1, owww_counts)):
    axes[0].text(count, bar.get_y() + bar.get_height()/2., 
                f' {int(count)}', va='center', fontsize=9, fontweight='bold')

# WWNW atoms
wwnw_contacts = {k: v for k, v in atom_type_contacts.items() if 'WWNW' in k}
wwnw_sorted = sorted(wwnw_contacts.items(), key=lambda x: x[1], reverse=True)[:15]
wwnw_names = [x[0].replace('(WWNW)', '') for x in wwnw_sorted]
wwnw_counts = [x[1] for x in wwnw_sorted]

colors_wwnw = plt.cm.Oranges(np.linspace(0.4, 0.9, len(wwnw_counts)))
bars2 = axes[1].barh(range(len(wwnw_names)), wwnw_counts, color=colors_wwnw, edgecolor='black', linewidth=1)

axes[1].set_yticks(range(len(wwnw_names)))
axes[1].set_yticklabels(wwnw_names, fontsize=11, fontweight='bold')
axes[1].set_xlabel('Total Contacts', fontsize=12, fontweight='bold')
axes[1].set_title('Top WWNW Atoms (Most Interactive)', fontsize=13, fontweight='bold', pad=15)
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

for i, (bar, count) in enumerate(zip(bars2, wwnw_counts)):
    axes[1].text(count, bar.get_y() + bar.get_height()/2., 
                f' {int(count)}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plot_file_3 = os.path.join(OUTPUT_DIR, "03_atoms_by_molecule.png")
plt.savefig(plot_file_3, dpi=300, bbox_inches='tight')
print(f"✅ Plot 3: {plot_file_3}")
plt.close()

# PLOT 4: Statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(contacts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(np.mean(contacts), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {np.mean(contacts):.1f}')
axes[0, 0].set_xlabel('Contacts per Frame', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Contact Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Box plot
axes[0, 1].boxplot(contacts, vert=True)
axes[0, 1].set_ylabel('Contacts per Frame', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Box Plot', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# Cumulative
cumulative = np.cumsum(contacts)
axes[1, 0].plot(frames, cumulative, color='darkgreen', linewidth=2)
axes[1, 0].fill_between(frames, cumulative, alpha=0.3, color='lightgreen')
axes[1, 0].set_xlabel('Frame Number', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Cumulative Contacts', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Cumulative Interactions', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Pie chart: top 10 pairs contribution
top_10_sum = sum([c for _, c in sorted_inter_pairs[:10]])
other_sum = total_contacts - top_10_sum

labels = [f"{p.split('-')[0]} ↔ {p.split('-')[1]}" for p, _ in sorted_inter_pairs[:10]]
labels.append("Others")

sizes = [c for _, c in sorted_inter_pairs[:10]] + [other_sum]
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(labels)))

wedges, texts, autotexts = axes[1, 1].pie(sizes, labels=labels, autopct='%1.1f%%', 
                                           colors=colors_pie, startangle=90, textprops={'fontsize': 8})
axes[1, 1].set_title('Contribution of Top 10 Pairs', fontsize=12, fontweight='bold')

plt.tight_layout()
plot_file_4 = os.path.join(OUTPUT_DIR, "04_interaction_statistics.png")
plt.savefig(plot_file_4, dpi=300, bbox_inches='tight')
print(f"✅ Plot 4: {plot_file_4}")
plt.close()

# PLOT 5: Heatmap of OWWW vs WWNW
fig, ax = plt.subplots(figsize=(10, 9))

# Get unique atom types by molecule
owww_types = sorted(list(owww_atom_names))
wwnw_types = sorted(list(wwnw_atom_names))

# Create matrix
interaction_matrix = np.zeros((len(owww_types), len(wwnw_types)))

for pair, count in intermolecular_pairs.items():
    owww_atom, wwnw_atom = pair.split('-')
    owww_atom = owww_atom.replace('(OWWW)', '')
    wwnw_atom = wwnw_atom.replace('(WWNW)', '')
    
    i = owww_types.index(owww_atom)
    j = wwnw_types.index(wwnw_atom)
    interaction_matrix[i, j] = count

sns.heatmap(interaction_matrix, 
            xticklabels=wwnw_types,
            yticklabels=owww_types,
            cmap='YlOrRd', 
            ax=ax,
            cbar_kws={'label': 'Contacts'},
            square=True,
            linewidths=0.3,
            annot=False)

ax.set_title('Interaction Matrix: OWWW (rows) vs WWNW (columns)', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('WWNW Atom Types', fontsize=12, fontweight='bold')
ax.set_ylabel('OWWW Atom Types', fontsize=12, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plot_file_5 = os.path.join(OUTPUT_DIR, "05_interaction_matrix_OWWW_vs_WWNW.png")
plt.savefig(plot_file_5, dpi=300, bbox_inches='tight')
print(f"✅ Plot 5: {plot_file_5}")
plt.close()

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*120)
print("ANALYSIS COMPLETE - WITH MOLECULE SOURCE INFORMATION")
print("="*120)
print(f"Total unique OWWW-WWNW interaction pairs: {len(intermolecular_pairs)}")
print(f"\nMost common interaction:")
pair, count = sorted_inter_pairs[0]
owww_atom, wwnw_atom = pair.split('-')
print(f"   {owww_atom} (OWWW) ↔ {wwnw_atom} (WWNW)")
print(f"   Contact count: {count}")

print(f"\nTop 5 most interactive atoms:")
for rank, (atom, count) in enumerate(sorted_atom_types[:5], 1):
    print(f"   {rank}. {atom}: {count} contacts")

print(f"\n✅ Output files saved:")
print(f"   - {output_file}")
print(f"   - {plot_file_1}")
print(f"   - {plot_file_2}")
print(f"   - {plot_file_3}")
print(f"   - {plot_file_4}")
print(f"   - {plot_file_5}")
print("="*120)







In [ ]:
# SAVE HEATMAP MATRIX DATA 
# ============================================================================

print("\n" + "="*120)
print("SAVING HEATMAP DATA...")
print("="*120)

# Import pandas for Excel export
import pandas as pd

# Create DataFrame from interaction matrix
df_matrix = pd.DataFrame(
    interaction_matrix,
    index=owww_types,
    columns=wwnw_types
)

# Print to console (readable format)
print("\nINTERACTION MATRIX (OWWW rows × WWNW columns):")
print("-" * 120)
print(df_matrix.to_string())
print("-" * 120)

# Save as Excel file
excel_file = os.path.join(OUTPUT_DIR, "interaction_matrix.xlsx")
df_matrix.to_excel(excel_file)
print(f"\n✅ Excel file saved: {excel_file}")

# Also save as CSV (for backup)
csv_file = os.path.join(OUTPUT_DIR, "interaction_matrix.csv")
df_matrix.to_csv(csv_file)
print(f"✅ CSV file saved: {csv_file}")

# Save matrix as text file with nice formatting
txt_matrix_file = os.path.join(OUTPUT_DIR, "interaction_matrix.txt")
with open(txt_matrix_file, "w") as f:
    f.write("INTERACTION MATRIX: OWWW (rows) vs WWNW (columns)\n")
    f.write("="*120 + "\n\n")
    f.write(df_matrix.to_string())
    f.write("\n\n" + "="*120 + "\n")
    f.write("\nMatrix Statistics:\n")
    f.write(f"Total interactions: {np.sum(interaction_matrix)}\n")
    f.write(f"Max interaction: {np.max(interaction_matrix)}\n")
    f.write(f"Min interaction: {np.min(interaction_matrix)}\n")
    f.write(f"Mean interaction: {np.mean(interaction_matrix):.2f}\n")

print(f"✅ Text matrix file saved: {txt_matrix_file}")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*120)
print("ANALYSIS COMPLETE - WITH MOLECULE SOURCE INFORMATION")
print("="*120)
print(f"Total unique OWWW-WWNW interaction pairs: {len(intermolecular_pairs)}")
print(f"\nMost common interaction:")
pair, count = sorted_inter_pairs[0]
owww_atom, wwnw_atom = pair.split('-')
print(f"   {owww_atom} (OWWW) ↔ {wwnw_atom} (WWNW)")
print(f"   Contact count: {count}")

print(f"\nTop 5 most interactive atoms:")
for rank, (atom, count) in enumerate(sorted_atom_types[:5], 1):
    print(f"   {rank}. {atom}: {count} contacts")

print(f"\n✅ Output files saved:")
print(f"   - {output_file}")
print(f"   - {plot_file_1}")
print(f"   - {plot_file_2}")
print(f"   - {plot_file_3}")
print(f"   - {plot_file_4}")
print(f"   - {plot_file_5}")
print(f"   - {excel_file} (HEATMAP MATRIX)")
print(f"   - {csv_file}")
print(f"   - {txt_matrix_file}")
print("="*120)

In [ ]:
# SAVE Plots with smaller fonts
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ============================================================================
# LOAD EXCEL FILE
# ============================================================================

OUTPUT_DIR = r".\data\output"
excel_file = os.path.join(OUTPUT_DIR, "interaction_matrix.xlsx")

print(f"Loading: {excel_file}\n")

# Load Excel with row names as index
df_matrix = pd.read_excel(excel_file, index_col=0)

# Convert to numpy array
interaction_matrix = df_matrix.values
owww_types = df_matrix.index.tolist()
wwnw_types = df_matrix.columns.tolist()

print(f"Matrix loaded: {len(owww_types)} × {len(wwnw_types)}")
print(f"Total interactions: {np.sum(interaction_matrix)}\n")

# ============================================================================
# PLOT 1: Large Readable Heatmap
# ============================================================================

print("Creating Plot 1: Large readable heatmap...")
fig, ax = plt.subplots(figsize=(16, 14))

sns.heatmap(interaction_matrix, 
            xticklabels=wwnw_types,
            yticklabels=owww_types,
            cmap='YlOrRd', 
            ax=ax,
            cbar_kws={'label': 'Number of Contacts', 'shrink': 0.8},
            square=True,
            linewidths=0.5,
            linecolor='gray',
            annot=False)

ax.set_title('Interaction Matrix: OWWW (rows) vs WWNW (columns)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('WWNW Atom Types', fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel('OWWW Atom Types', fontsize=14, fontweight='bold', labelpad=10)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plot_file1 = os.path.join(OUTPUT_DIR, "01_heatmap_READABLE_large.png")
plt.savefig(plot_file1, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file1}\n")
plt.close()

# ============================================================================
# PLOT 2: With Values in Cells
# ============================================================================

print("Creating Plot 2: Heatmap with values...")
fig, ax = plt.subplots(figsize=(14, 12))

sns.heatmap(interaction_matrix, 
            xticklabels=wwnw_types,
            yticklabels=owww_types,
            cmap='YlOrRd', 
            ax=ax,
            cbar_kws={'label': 'Number of Contacts'},
            square=True,
            linewidths=0.5,
            linecolor='gray',
            annot=True,
            fmt='g',
            annot_kws={'size': 7})

ax.set_title('Interaction Matrix with Values (OWWW × WWNW)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('WWNW Atom Types', fontsize=13, fontweight='bold')
ax.set_ylabel('OWWW Atom Types', fontsize=13, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plot_file2 = os.path.join(OUTPUT_DIR, "02_heatmap_WITH_VALUES.png")
plt.savefig(plot_file2, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file2}\n")
plt.close()

# ============================================================================
# PLOT 3: Clustered Heatmap
# ============================================================================

print("Creating Plot 3: Clustered heatmap...")
try:
    g = sns.clustermap(df_matrix,
                       cmap='YlOrRd',
                       figsize=(14, 12),
                       cbar_kws={'label': 'Number of Contacts'},
                       xticklabels=True,
                       yticklabels=True,
                       linewidths=0.5,
                       linecolor='gray')
    
    g.ax_heatmap.set_title('Clustered Interaction Matrix', 
                           fontsize=16, fontweight='bold', pad=20)
    g.ax_heatmap.set_xlabel('WWNW Atom Types', fontsize=13, fontweight='bold')
    g.ax_heatmap.set_ylabel('OWWW Atom Types', fontsize=13, fontweight='bold')
    
    g.ax_heatmap.tick_params(axis='x', labelsize=9, rotation=45)
    g.ax_heatmap.tick_params(axis='y', labelsize=9, rotation=0)
    
    plot_file3 = os.path.join(OUTPUT_DIR, "03_heatmap_CLUSTERED.png")
    plt.savefig(plot_file3, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {plot_file3}\n")
    plt.close()
except Exception as e:
    print(f"⚠️ Clustering failed: {e}\n")

# ============================================================================
# PLOT 4: Minimal Overview
# ============================================================================

print("Creating Plot 4: Minimal overview...")
fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(interaction_matrix, cmap='YlOrRd', aspect='auto')

ax.set_xticks(np.arange(len(wwnw_types)))
ax.set_yticks(np.arange(len(owww_types)))
ax.set_xticklabels(wwnw_types, fontsize=8, rotation=45, ha='right')
ax.set_yticklabels(owww_types, fontsize=8)

ax.set_title('Interaction Matrix Overview', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('WWNW Atom Types', fontsize=11, fontweight='bold')
ax.set_ylabel('OWWW Atom Types', fontsize=11, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, label='Number of Contacts')

plt.tight_layout()
plot_file4 = os.path.join(OUTPUT_DIR, "04_heatmap_MINIMAL.png")
plt.savefig(plot_file4, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file4}\n")
plt.close()

# ============================================================================
# PLOT 5: Top Interactions Bar Chart
# ============================================================================

print("Creating Plot 5: Top interactions bar chart...")

# Flatten matrix and get top interactions
interactions = []
for i, owww in enumerate(owww_types):
    for j, wwnw in enumerate(wwnw_types):
        if interaction_matrix[i, j] > 0:
            interactions.append({
                'pair': f"{owww}-{wwnw}",
                'count': interaction_matrix[i, j]
            })

# Sort and get top 30
interactions_sorted = sorted(interactions, key=lambda x: x['count'], reverse=True)[:30]
pairs = [x['pair'] for x in interactions_sorted]
counts = [x['count'] for x in interactions_sorted]

fig, ax = plt.subplots(figsize=(12, 10))

colors = plt.cm.YlOrRd(np.linspace(0.4, 0.9, len(counts)))
bars = ax.barh(range(len(pairs)), counts, color=colors, edgecolor='black', linewidth=0.7)

ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel('Number of Contacts', fontsize=12, fontweight='bold')
ax.set_title('Top 30 Atom Pair Interactions', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# Add value labels
for i, (bar, count) in enumerate(zip(bars, counts)):
    ax.text(count, bar.get_y() + bar.get_height()/2., 
            f' {int(count)}', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plot_file5 = os.path.join(OUTPUT_DIR, "05_top_30_pairs.png")
plt.savefig(plot_file5, dpi=300, bbox_inches='tight')
print(f"✅ Saved: {plot_file5}\n")
plt.close()





# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*100)
print("HEATMAP PLOTS CREATED FROM EXCEL FILE")
print("="*100)
print(f"\nMatrix Statistics:")
print(f"  OWWW atom types: {len(owww_types)}")
print(f"  WWNW atom types: {len(wwnw_types)}")
print(f"  Total interactions: {np.sum(interaction_matrix)}")
print(f"  Max interaction: {np.max(interaction_matrix)}")
print(f"  Min interaction: {np.min(interaction_matrix)}")
print(f"  Mean interaction: {np.mean(interaction_matrix):.2f}")
print(f"  Median interaction: {np.median(interaction_matrix):.2f}")

# Find most common pair
max_idx = np.unravel_index(np.argmax(interaction_matrix), interaction_matrix.shape)
print(f"\nMost common interaction:")
print(f"  {owww_types[max_idx[0]]} ↔ {wwnw_types[max_idx[1]]}")
print(f"  Count: {interaction_matrix[max_idx]}")

print(f"\n✅ All plots saved to: {OUTPUT_DIR}")
print("="*100)

In [ ]:
# Script calculates Interaction HEATMAP MATRIX without Hydrogen atom representation between two different molecule types in a simuulation box
# The abbreviation "OWWW" stands for C8-WWWDK and "WWNW" stands for C8-WWNWK.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

OUTPUT_DIR = r".\data\output_noHbond"

# Load Excel
print("Loading Excel file...")
df = pd.read_excel(os.path.join(OUTPUT_DIR, "interaction_matrix.xlsx"), index_col=0)
data = df.values
owww_types = df.index.tolist()
wwnw_types = df.columns.tolist()

# Filter non-H atoms
non_h_owww = [a for a in owww_types if not a.startswith('H')]
non_h_wwnw = [a for a in wwnw_types if not a.startswith('H')]

print(f"Original: {len(owww_types)} × {len(wwnw_types)}")
print(f"Filtered: {len(non_h_owww)} × {len(non_h_wwnw)}\n")

# Create filtered matrix and DataFrame
filtered_data = df.loc[non_h_owww, non_h_wwnw].values
df_filtered = df.loc[non_h_owww, non_h_wwnw]

# ============================================================================
# EXTRACT TOP INTERACTIONS FOR BAR PLOTS
# ============================================================================

print("Extracting top interactions...")
interactions = []
for i, owww in enumerate(non_h_owww):
    for j, wwnw in enumerate(non_h_wwnw):
        if filtered_data[i, j] > 0:
            interactions.append({
                'pair': f"{owww}-{wwnw}",
                'owww': owww,
                'wwnw': wwnw,
                'count': filtered_data[i, j]
            })

# Sort by count
interactions_sorted = sorted(interactions, key=lambda x: x['count'], reverse=True)

print(f"Total unique pairs: {len(interactions)}\n")

# ============================================================================
# PLOT 1: Heatmap With Values
# ============================================================================

print("Creating Plot 1: Heatmap with values...")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(filtered_data, 
            xticklabels=non_h_wwnw,
            yticklabels=non_h_owww,
            cmap='YlOrRd', 
            annot=True,
            fmt='g',
            square=True,
            ax=ax,
            cbar_kws={'label': 'Contacts'})
ax.set_title('Non-Hydrogen Atoms Only', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_heatmap_NO_H_values.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 01_heatmap_NO_H_values.png")
plt.close()

# ============================================================================
# PLOT 2: Clean Heatmap
# ============================================================================

print("Creating Plot 2: Clean heatmap...")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(filtered_data, 
            xticklabels=non_h_wwnw,
            yticklabels=non_h_owww,
            cmap='YlOrRd', 
            square=True,
            ax=ax,
            cbar_kws={'label': 'Contacts'})
ax.set_title('Non-Hydrogen Atoms Only (Clean)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_heatmap_NO_H_clean.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 02_heatmap_NO_H_clean.png")
plt.close()

# ============================================================================
# PLOT 3: Clustered Heatmap
# ============================================================================

print("Creating Plot 3: Clustered heatmap...")
g = sns.clustermap(df_filtered,
                   cmap='YlOrRd',
                   figsize=(12, 10),
                   square=True)
g.ax_heatmap.set_title('Non-Hydrogen Atoms (Clustered)', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, "03_heatmap_NO_H_clustered.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 03_heatmap_NO_H_clustered.png")
plt.close()

# ============================================================================
# PLOT 4: Top 40 Interactions Bar Chart (Horizontal)
# ============================================================================

print("Creating Plot 4: Top 40 interactions bar chart...")
top_40 = interactions_sorted[:40]
pairs = [x['pair'] for x in top_40]
counts = [x['count'] for x in top_40]

fig, ax = plt.subplots(figsize=(12, 14))

colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(counts)))
bars = ax.barh(range(len(pairs)), counts, color=colors, edgecolor='black', linewidth=0.7)

ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel('Number of Contacts', fontsize=12, fontweight='bold')
ax.set_title('Top 40 Non-Hydrogen Atom Pair Interactions', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# Add value labels
for i, (bar, count) in enumerate(zip(bars, counts)):
    ax.text(count, bar.get_y() + bar.get_height()/2., 
            f' {int(count)}', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_top_40_pairs_bar_chart.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 04_top_40_pairs_bar_chart.png")
plt.close()

# ============================================================================
# PLOT 5: Top 20 - HORIZONTAL with Molecule Labels
# ============================================================================

print("Creating Plot 5: Top 20 horizontal with molecule labels...")
top_20 = interactions_sorted[:20]
pairs_20 = [x['pair'] for x in top_20]
counts_20 = [x['count'] for x in top_20]
owww_20 = [x['owww'] for x in top_20]
wwnw_20 = [x['wwnw'] for x in top_20]

# Create labels
labels_with_molecules = [f"{owww}(OWWW) → {wwnw}(WWNW)" 
                         for owww, wwnw in zip(owww_20, wwnw_20)]

fig, ax = plt.subplots(figsize=(12, 12))

colors_20 = plt.cm.YlOrRd(np.linspace(0.35, 0.95, len(counts_20)))
bars = ax.barh(range(len(counts_20)), counts_20, color=colors_20, 
               edgecolor='black', linewidth=1, alpha=0.88)

ax.set_yticks(range(len(labels_with_molecules)))
ax.set_yticklabels(labels_with_molecules, fontsize=10, fontweight='bold')
ax.set_xlabel('Number of Contacts', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Non-Hydrogen Atom Interactions\nOWWW → WWNW', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# Add value labels and rank
for i, (bar, count) in enumerate(zip(bars, counts_20)):
    # Count label
    ax.text(count + max(counts_20)*0.01, bar.get_y() + bar.get_height()/2.,
            f'{int(count)}',
            va='center', fontsize=9, fontweight='bold')
    # Rank label
    #ax.text(-max(counts_20)*0.02, bar.get_y() + bar.get_height()/2.,
    #        f'#{i+1}',
    #        va='center', ha='right', fontsize=10, fontweight='bold', color='darkred')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_top_20_pairs_HORIZONTAL_molecules.png"), 
            dpi=300, bbox_inches='tight')
print("✅ Saved: 05_top_20_pairs_HORIZONTAL_molecules.png")
plt.close()
# ============================================================================
# PLOT 6: Top OWWW Atoms (by total contacts)
# ============================================================================

print("Creating Plot 6: Top OWWW atoms...")
owww_totals = {}
for inter in interactions_sorted:
    owww = inter['owww']
    count = inter['count']
    owww_totals[owww] = owww_totals.get(owww, 0) + count

owww_sorted = sorted(owww_totals.items(), key=lambda x: x[1], reverse=True)[:15]
owww_names = [x[0] for x in owww_sorted]
owww_counts = [x[1] for x in owww_sorted]

fig, ax = plt.subplots(figsize=(12, 8))

colors_owww = plt.cm.Blues(np.linspace(0.4, 0.9, len(owww_counts)))
bars = ax.barh(range(len(owww_names)), owww_counts, color=colors_owww, edgecolor='black', linewidth=1)

ax.set_yticks(range(len(owww_names)))
ax.set_yticklabels(owww_names, fontsize=11, fontweight='bold')
ax.set_xlabel('Total Contacts', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Most Interactive OWWW Atoms (Non-H)', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

for i, (bar, count) in enumerate(zip(bars, owww_counts)):
    ax.text(count, bar.get_y() + bar.get_height()/2., 
            f' {int(count)}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_top_15_OWWW_atoms.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 06_top_15_OWWW_atoms.png")
plt.close()

# ============================================================================
# PLOT 7: Top WWNW Atoms (by total contacts)
# ============================================================================

print("Creating Plot 7: Top WWNW atoms...")
wwnw_totals = {}
for inter in interactions_sorted:
    wwnw = inter['wwnw']
    count = inter['count']
    wwnw_totals[wwnw] = wwnw_totals.get(wwnw, 0) + count

wwnw_sorted = sorted(wwnw_totals.items(), key=lambda x: x[1], reverse=True)[:15]
wwnw_names = [x[0] for x in wwnw_sorted]
wwnw_counts = [x[1] for x in wwnw_sorted]

fig, ax = plt.subplots(figsize=(12, 8))

colors_wwnw = plt.cm.Oranges(np.linspace(0.4, 0.9, len(wwnw_counts)))
bars = ax.barh(range(len(wwnw_names)), wwnw_counts, color=colors_wwnw, edgecolor='black', linewidth=1)

ax.set_yticks(range(len(wwnw_names)))
ax.set_yticklabels(wwnw_names, fontsize=11, fontweight='bold')
ax.set_xlabel('Total Contacts', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Most Interactive WWNW Atoms (Non-H)', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

for i, (bar, count) in enumerate(zip(bars, wwnw_counts)):
    ax.text(count, bar.get_y() + bar.get_height()/2., 
            f' {int(count)}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_top_15_WWNW_atoms.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 07_top_15_WWNW_atoms.png")
plt.close()

# ============================================================================
# PLOT 8: Cumulative Interactions
# ============================================================================

print("Creating Plot 8: Cumulative interactions...")
cumulative_counts = np.cumsum([x['count'] for x in interactions_sorted])
total_interactions = cumulative_counts[-1]
cumulative_percent = (cumulative_counts / total_interactions) * 100

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(range(1, len(cumulative_percent) + 1), cumulative_percent, 
        linewidth=2.5, color='darkblue', marker='o', markersize=4, markevery=5)
ax.fill_between(range(1, len(cumulative_percent) + 1), cumulative_percent, 
                alpha=0.3, color='lightblue')

ax.axhline(y=50, color='r', linestyle='--', linewidth=1.5, label='50% cumulative')
ax.axhline(y=80, color='orange', linestyle='--', linewidth=1.5, label='80% cumulative')
ax.axhline(y=90, color='green', linestyle='--', linewidth=1.5, label='90% cumulative')

ax.set_xlabel('Pair Rank', fontsize=12, fontweight='bold')
ax.set_ylabel('Cumulative Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('Cumulative Percentage of Total Interactions (Non-H)', fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_cumulative_interactions.png"), dpi=300, bbox_inches='tight')
print("✅ Saved: 08_cumulative_interactions.png")
plt.close()

# ============================================================================
# SAVE TO EXCEL
# ============================================================================

print("\nSaving to Excel...")

# Save filtered matrix
excel_no_h = os.path.join(OUTPUT_DIR, "interaction_matrix_NO_HYDROGENS.xlsx")
df_filtered.to_excel(excel_no_h)
print(f"✅ Saved: interaction_matrix_NO_HYDROGENS.xlsx")

# Save ranked pairs
excel_ranking = os.path.join(OUTPUT_DIR, "top_interactions_NO_HYDROGENS.xlsx")
ranking_df = pd.DataFrame({
    'Rank': range(1, len(interactions_sorted) + 1),
    'OWWW_Atom': [x['owww'] for x in interactions_sorted],
    'WWNW_Atom': [x['wwnw'] for x in interactions_sorted],
    'Interaction_Pair': [x['pair'] for x in interactions_sorted],
    'Contact_Count': [x['count'] for x in interactions_sorted],
    'Percentage': [f"{(x['count']/total_interactions)*100:.2f}%" for x in interactions_sorted]
})
ranking_df.to_excel(excel_ranking, index=False)
print(f"✅ Saved: top_interactions_NO_HYDROGENS.xlsx")

# ============================================================================
# PRINT SUMMARY
# ============================================================================

print("\n" + "="*100)
print("ANALYSIS SUMMARY - NON-HYDROGEN ATOMS")
print("="*100)
print(f"\nMatrix Dimensions:")
print(f"  Original: {len(owww_types)} × {len(wwnw_types)}")
print(f"  Filtered (Non-H): {len(non_h_owww)} × {len(non_h_wwnw)}")

print(f"\nInteraction Statistics:")
print(f"  Total interactions: {np.sum(filtered_data)}")
print(f"  Max interaction: {np.max(filtered_data)}")
print(f"  Mean interaction: {np.mean(filtered_data):.2f}")
print(f"  Total unique pairs: {len(interactions)}")

print(f"\nTop 5 Interactions:")
for rank, inter in enumerate(interactions_sorted[:5], 1):
    percent = (inter['count'] / total_interactions) * 100
    print(f"  {rank}. {inter['pair']}: {inter['count']} contacts ({percent:.2f}%)")

print(f"\nTop 5 OWWW Atoms:")
for rank, (atom, count) in enumerate(owww_sorted[:5], 1):
    print(f"  {rank}. {atom}: {count} total contacts")

print(f"\nTop 5 WWNW Atoms:")
for rank, (atom, count) in enumerate(wwnw_sorted[:5], 1):
    print(f"  {rank}. {atom}: {count} total contacts")

print(f"\n✅ All files saved to: {OUTPUT_DIR}")
print("="*100)